# CS336 Spring 2026 - Profile and Optimize BPE Training
----
本节目标只有一个：

> **不改变 BPE 训练结果，让上一节的正确版本更快。**

路线：

```text
正确的慢版本
→ 测量哪里慢
→ 发现一次 merge 只影响少量 pre-token
→ 建立 pair → 受影响 sequence 的反向索引
→ 只更新变化部分
→ 与慢版本逐轮对答案
→ 确认正确后再测性能
```

术语对照：

| 英文术语 | 中文理解 |
|---|---|
| reference implementation / oracle | 正确但较慢、用来当标准答案的实现 |
| profiling | 测量时间具体花在哪里 |
| merge locality | 一次 merge 只影响局部数据 |
| inverted index | 从 pair 反查哪些 sequence 包含它 |
| incremental update | 不全部重算，只更新变化部分 |
| differential testing | 新旧实现跑同一输入并比较结果 |
| bottleneck | 当前最限制性能的部分 |

> 本节先有可信的 oracle, 再优化; 先证明结果相同, 再讨论快多少

## 1. 为什么保留“正确的慢版本”？

优化代码会变复杂，最危险的情况是：

```text
程序能运行
结果看起来合理
但边界情况已经悄悄算错
```

所以保留上一节的简单实现作为“标准答案”。

以后优化版必须满足：

```text
相同输入
→ ordered merges 完全相同
→ final pre-token state 完全相同
```

这就是 reference implementation / correctness oracle 的作用。

In [ ]:
from __future__ import annotations

import random
from collections import Counter, defaultdict
from dataclasses import dataclass
from time import perf_counter
from typing import TypeAlias

Token: TypeAlias = bytes
TokenSequence: TypeAlias = tuple[Token, ...]
PretokenCounts: TypeAlias = Counter[TokenSequence]
Pair: TypeAlias = tuple[Token, Token]


def local_pair_counts(sequence: TokenSequence) -> Counter[Pair]:
    return Counter(zip(sequence, sequence[1:]))


def count_adjacent_pairs(pretoken_counts: PretokenCounts) -> Counter[Pair]:
    result: Counter[Pair] = Counter()
    for sequence, frequency in pretoken_counts.items():
        for pair, local_count in local_pair_counts(sequence).items():
            result[pair] += local_count * frequency
    return result


def select_best_pair(pair_counts: Counter[Pair]) -> Pair | None:
    if not pair_counts:
        return None
    return max(pair_counts, key=lambda pair: (pair_counts[pair], pair))


def merge_pair_in_sequence(sequence: TokenSequence, pair: Pair) -> TokenSequence:
    left, right = pair
    merged = left + right
    output = []
    i = 0
    while i < len(sequence):
        if i + 1 < len(sequence) and sequence[i] == left and sequence[i + 1] == right:
            output.append(merged)
            i += 2
        else:
            output.append(sequence[i])
            i += 1
    return tuple(output)


def apply_merge_reference(
    pretoken_counts: PretokenCounts, pair: Pair
) -> PretokenCounts:
    updated: PretokenCounts = Counter()
    for sequence, frequency in pretoken_counts.items():
        updated[merge_pair_in_sequence(sequence, pair)] += frequency
    return updated


def train_reference(
    initial_counts: PretokenCounts, num_merges: int
) -> tuple[list[Pair], PretokenCounts]:
    counts = Counter(initial_counts)
    merges = []
    for _ in range(num_merges):
        pair_counts = count_adjacent_pairs(counts)
        pair = select_best_pair(pair_counts)
        if pair is None:
            break
        merges.append(pair)
        counts = apply_merge_reference(counts, pair)
    return merges, counts


## 2. 慢版本为什么慢？

假设这一轮 winner 是：

```text
(b"t", b"h")
```

只有包含 `t | h` 的 pre-token 会改变，例如：

```text
t | h | e
t | h | e | r | e
```

`cat`、`dog` 等完全不会改变。

但慢版本每轮仍然：

```text
扫描全部 sequence
→ 重算全部 pair
→ 选 winner
→ 再扫描全部 sequence 做 merge
```

所以核心浪费是：

> **程序不知道哪些 sequence 真正受本轮 merge 影响，只能全部重算。**

## 3. 构造固定 benchmark 数据

这里先绕过 regex pre-tokenization，直接生成 `PretokenCounts`。

目的：单独测 BPE merge loop，而不是把 regex、I/O 和 merge loop 混成一个时间。

In [2]:
def make_synthetic_counts(num_sequences: int = 900, seed: int = 336) -> PretokenCounts:
    rng = random.Random(seed)
    alphabet = [bytes([x]) for x in range(ord("a"), ord("z") + 1)]
    counts: PretokenCounts = Counter()

    while len(counts) < num_sequences:
        length = rng.randint(6, 16)
        sequence = tuple(rng.choice(alphabet) for _ in range(length))
        counts[sequence] += rng.randint(1, 50)

    return counts


benchmark_counts = make_synthetic_counts()

print("不同 pre-token 数:", len(benchmark_counts))
print("考虑 frequency 后的总出现次数:", sum(benchmark_counts.values()))
print("distinct state 中的 token 位置总数:", sum(len(s) for s in benchmark_counts))

不同 pre-token 数: 900
考虑 frequency 后的总出现次数: 22639
distinct state 中的 token 位置总数: 9883


## 4. 先测慢版本

性能实验至少要保证相同输入并重复运行。这里重复 3 次，取中位数。

不要把一次机器上的秒数当成普遍结论；我们真正关心的是优化前后在相同条件下的变化。

In [3]:
def time_call(fn, *args, repeats=3, **kwargs):
    durations = []
    result = None
    for _ in range(repeats):
        start = perf_counter()
        result = fn(*args, **kwargs)
        durations.append(perf_counter() - start)
    durations.sort()
    return durations[len(durations) // 2], result


NUM_MERGES = 40

reference_time, reference_result = time_call(
    train_reference, benchmark_counts, NUM_MERGES
)

print("慢版本中位时间:", round(reference_time, 4), "秒")
print("完成 merge 数:", len(reference_result[0]))

慢版本中位时间: 0.1788 秒
完成 merge 数: 40


## 5. Profiling：要实际测量出哪里慢

把每轮拆成：

```text
A. 重新统计全部 pair
B. 从 pair_counts 中选择 winner
C. 扫描全部 sequence 执行 merge
```

分别计时。这样才能知道第一刀应该优化哪里。

In [4]:
@dataclass
class ReferenceProfile:
    pair_count_seconds: float = 0.0
    winner_seconds: float = 0.0
    merge_seconds: float = 0.0


def profile_reference(initial_counts, num_merges):
    counts = Counter(initial_counts)
    p = ReferenceProfile()

    for _ in range(num_merges):
        start = perf_counter()
        pair_counts = count_adjacent_pairs(counts)
        p.pair_count_seconds += perf_counter() - start

        start = perf_counter()
        pair = select_best_pair(pair_counts)
        p.winner_seconds += perf_counter() - start
        if pair is None:
            break

        start = perf_counter()
        counts = apply_merge_reference(counts, pair)
        p.merge_seconds += perf_counter() - start

    return p


p = profile_reference(benchmark_counts, NUM_MERGES)
total = p.pair_count_seconds + p.winner_seconds + p.merge_seconds

for name, value in [
    ("重新统计全部 pair", p.pair_count_seconds),
    ("选择 winner", p.winner_seconds),
    ("扫描全部 sequence 做 merge", p.merge_seconds),
]:
    print(f"{name}: {value:.4f} 秒 ({100 * value / total:.1f}%)")


重新统计全部 pair: 0.1291 秒 (71.9%)
选择 winner: 0.0044 秒 (2.5%)
扫描全部 sequence 做 merge: 0.0460 秒 (25.6%)


## 6. 最关键的观察：一次 merge 具有局部性

考虑：

```text
x | a | b | y
```

merge `(a,b)` 后：

```text
x | ab | y
```

附近 pair 从：

```text
(x,a), (a,b), (b,y)
```

变成：

```text
(x,ab), (ab,y)
```

而不包含 `(a,b)` 的 sequence 完全不变。

所以我们真正需要快速回答：

```text
pair (a,b)
→ 哪些 distinct sequence 包含它？
```

这就是 **反向索引（inverted index）**：

$$
I(p)=\{w: p\text{ 出现在 }w\text{ 中}\}
$$

winner 选出后，只访问 `I(winner)`。

In [5]:
def build_pair_index(pretoken_counts: PretokenCounts) -> dict[Pair, set[TokenSequence]]:
    index = defaultdict(set)

    for sequence in pretoken_counts:
        for pair in local_pair_counts(sequence):
            index[pair].add(sequence)

    return dict(index)


demo = Counter(
    {
        (b"a", b"b", b"c"): 3,
        (b"x", b"a", b"b"): 2,
        (b"d", b"e"): 5,
    }
)

print("(a,b) 影响的 sequence:")
for sequence in build_pair_index(demo)[(b"a", b"b")]:
    print(sequence)


(a,b) 影响的 sequence:
(b'x', b'a', b'b')
(b'a', b'b', b'c')


## 7. `pair_counts` 和 `pair_to_sequences` 

这是本节最重要的区分之一。

### `pair_counts`

回答：

> 这个 pair 在整个训练 corpus 中加权后出现多少次？

例如：

```text
a | a | a | a
frequency = 7
```

`(a,a)` 在一个 sequence 内出现 3 次，因此全局贡献是：

$$
3\times7=21
$$

### `pair_to_sequences`

回答：

> 如果 merge 这个 pair，哪些 distinct sequence 需要更新？

上面的 sequence 在 set 中只需要出现一次。

所以：

```text
pretoken_counts:
sequence → corpus frequency

pair_counts:
pair → weighted global frequency

pair_to_sequences:
pair → affected distinct sequences
```

三个数据结构职责完全不同。

## 8. 增量更新的核心公式

某个受影响 sequence 的 corpus frequency 是 $f$。

merge 前，它对 pair $p$ 的局部出现次数为 $c_{old}(p)$；merge 后为 $c_{new}(p)$。

那么全局 pair count 更新为：

$$
C_{new}(p)
=
C_{old}(p)
-
f c_{old}(p)
+
f c_{new}(p)
$$

直觉就是：

```text
撤销旧 sequence 的贡献
→ merge
→ 加入新 sequence 的贡献
```

这就是“增量更新”：不重算所有没变的数据。

In [6]:
def subtract_sequence_contribution(
    sequence,
    frequency,
    pair_counts,
    pair_to_sequences,
):
    for pair, local_count in local_pair_counts(sequence).items():
        pair_counts[pair] -= local_count * frequency

        if pair_counts[pair] == 0:
            del pair_counts[pair]

        pair_to_sequences[pair].discard(sequence)
        if not pair_to_sequences[pair]:
            del pair_to_sequences[pair]


def add_sequence_contribution(
    sequence,
    frequency,
    pair_counts,
    pair_to_sequences,
):
    for pair, local_count in local_pair_counts(sequence).items():
        pair_counts[pair] += local_count * frequency
        pair_to_sequences.setdefault(pair, set()).add(sequence)


@dataclass
class IncrementalState:
    pretoken_counts: PretokenCounts
    pair_counts: Counter[Pair]
    pair_to_sequences: dict[Pair, set[TokenSequence]]


def initialize_incremental_state(initial_counts):
    counts = Counter(initial_counts)
    return IncrementalState(
        pretoken_counts=counts,
        pair_counts=count_adjacent_pairs(counts),
        pair_to_sequences=build_pair_index(counts),
    )


## 9. 隐藏难点：merge 后的 sequence 可能“撞车”

两个不同旧 sequence：

```text
A，frequency = 7
B，frequency = 5
```

执行 winner 后可能都得到同一个 `new_sequence`。

那么新 frequency 必须是：

$$
7+5=12
$$

而且 `new_sequence` 甚至可能原本就已经存在。

所以不能简单写：

```python
counts[new_sequence] = frequency
```

必须先聚合 frequency，并保证 `pair_counts` 与反向索引也同步更新。

In [7]:
def apply_merge_incremental(state: IncrementalState, pair: Pair) -> int:
    # 做快照：后面会修改真正的 index。
    affected = set(state.pair_to_sequences.get(pair, set()))
    transformed: PretokenCounts = Counter()

    # 先删除所有受影响的旧状态。
    for old_sequence in affected:
        frequency = state.pretoken_counts.pop(old_sequence)

        subtract_sequence_contribution(
            old_sequence,
            frequency,
            state.pair_counts,
            state.pair_to_sequences,
        )

        new_sequence = merge_pair_in_sequence(old_sequence, pair)
        transformed[new_sequence] += frequency

    # 再把 merge 后的状态聚合回去。
    for new_sequence, added_frequency in transformed.items():
        existing_frequency = state.pretoken_counts.get(new_sequence, 0)

        if existing_frequency:
            subtract_sequence_contribution(
                new_sequence,
                existing_frequency,
                state.pair_counts,
                state.pair_to_sequences,
            )

        combined_frequency = existing_frequency + added_frequency

        state.pretoken_counts[new_sequence] = combined_frequency

        add_sequence_contribution(
            new_sequence,
            combined_frequency,
            state.pair_counts,
            state.pair_to_sequences,
        )

    return len(affected)


## 10. 为什么先验证状态，而不是先看速度？

增量算法最怕 metadata 悄悄失真。

所以调试时从真实的 `pretoken_counts` 重新完整计算：

```text
expected pair_counts
expected pair_to_sequences
```

然后与维护中的状态比较。

只要不一致，立即失败。

这相当于每轮都问：

> “如果我现在从零重算，结果是否与增量维护的一模一样？”

In [8]:
def validate_state(state: IncrementalState) -> None:
    expected_pairs = count_adjacent_pairs(state.pretoken_counts)
    expected_index = build_pair_index(state.pretoken_counts)

    assert state.pair_counts == expected_pairs
    assert state.pair_to_sequences == expected_index


small = Counter(
    {
        (b"t", b"h", b"e"): 3,
        (b" ", b"t", b"h", b"e"): 2,
        (b"c", b"a", b"t"): 4,
    }
)

state = initialize_incremental_state(small)
winner = select_best_pair(state.pair_counts)
affected = apply_merge_incremental(state, winner)
validate_state(state)

print("本轮 winner:", winner)
print("真正受影响的 distinct sequence 数:", affected)
print("增量状态验证: 通过")

本轮 winner: (b't', b'h')
真正受影响的 distinct sequence 数: 2
增量状态验证: 通过


## 11. 完整增量训练器

每轮现在变成：

```text
从 pair_counts 选 winner
→ 从反向索引找到 affected sequences
→ 只更新这些 sequences
```

目前 `select_best_pair()` 仍然线性扫描所有 current pairs。

这是故意的：本节只优化一个核心问题，避免同时引入 heap 后难以定位 correctness bug。

In [9]:
@dataclass
class IncrementalProfile:
    winner_seconds: float = 0.0
    update_seconds: float = 0.0
    rounds: int = 0
    total_affected: int = 0


def train_incremental(initial_counts, num_merges, *, validate_every_round=False):
    state = initialize_incremental_state(initial_counts)
    merges = []
    profile = IncrementalProfile()

    for _ in range(num_merges):
        start = perf_counter()
        pair = select_best_pair(state.pair_counts)
        profile.winner_seconds += perf_counter() - start

        if pair is None:
            break

        start = perf_counter()
        affected = apply_merge_incremental(state, pair)
        profile.update_seconds += perf_counter() - start

        merges.append(pair)
        profile.rounds += 1
        profile.total_affected += affected

        if validate_every_round:
            validate_state(state)

    return merges, state.pretoken_counts, profile


## 12. 差分测试：优化版必须和慢版本逐项相同

相同输入分别交给两个实现：

```text
reference
incremental
```

必须同时满足：

```text
ordered merges 完全相同
final pre-token state 完全相同
```

为什么 merge 顺序必须完全一样？

因为 BPE 有顺序依赖：

```text
某一轮 winner 不同
→ segmentation 不同
→ 下一轮 pair_counts 不同
→ 后续训练历史可能全部分叉
```

这种“两个实现对同一输入互相对答案”的测试方法叫 **differential testing（差分测试）**。

In [10]:
ref_merges, ref_final = train_reference(benchmark_counts, NUM_MERGES)
inc_merges, inc_final, _ = train_incremental(
    benchmark_counts, NUM_MERGES, validate_every_round=True
)

assert inc_merges == ref_merges
assert inc_final == ref_final

print("差分测试: 通过")
print("逐项比较 merge 数:", len(ref_merges))

差分测试: 通过
逐项比较 merge 数: 40


## 13. 为什么 benchmark 时关闭逐轮验证？

`validate_state()` 自己会重新扫描全部 state：

```text
重算全部 pair_counts
重建全部反向索引
```

它是昂贵的 correctness 检查。

因此：

```text
开发 / 调试：
validate_every_round=True

性能测试 / 正式 hot path：
validate_every_round=False
```

不要让 debug 检查污染性能数字。

In [11]:
incremental_time, incremental_result = time_call(
    train_incremental,
    benchmark_counts,
    NUM_MERGES,
    repeats=3,
    validate_every_round=False,
)

assert incremental_result[0] == reference_result[0]

speedup = reference_time / incremental_time

print("慢版本中位时间:", round(reference_time, 4), "秒")
print("增量版本中位时间:", round(incremental_time, 4), "秒")
print("本次实验观察到的加速:", round(speedup, 2), "x")
print("两版 merge history 一致: 通过")

慢版本中位时间: 0.1788 秒
增量版本中位时间: 0.0203 秒
本次实验观察到的加速: 8.8 x
两版 merge history 一致: 通过


## 14. 为什么会加速？

慢版本每轮处理所有 distinct sequence，记作 $W_t$。

增量版每轮只处理 winner 真正出现的 sequence：

$$
A_t = |I(p_t^*)|
$$

如果：

$$
A_t \ll W_t
$$

就能省掉大量无关工作。

但当前 winner selection 仍然扫描所有 current pairs；若第 $t$ 轮有 $P_t$ 个 distinct pairs，当前版本可以粗略理解为：

$$
O(N)+
\sum_t
\left[
O(P_t)+
\text{更新 }A_t\text{ 个 sequence 的工作}
\right]
$$

其中 $O(N)$ 是初始化 pair counts 与反向索引的一次完整扫描。

In [12]:
_, _, optimized_profile = train_incremental(benchmark_counts, NUM_MERGES)

total = optimized_profile.winner_seconds + optimized_profile.update_seconds

for name, value in [
    ("扫描 pair_counts 选 winner", optimized_profile.winner_seconds),
    ("增量更新 affected sequences", optimized_profile.update_seconds),
]:
    print(f"{name}: {value:.4f} 秒 ({100 * value / total:.1f}%)")

print(
    "每轮平均受影响 sequence 数:",
    round(optimized_profile.total_affected / optimized_profile.rounds, 2),
)
print("初始 distinct sequence 总数:", len(benchmark_counts))


扫描 pair_counts 选 winner: 0.0051 秒 (25.5%)
增量更新 affected sequences: 0.0150 秒 (74.5%)
每轮平均受影响 sequence 数: 18.73
初始 distinct sequence 总数: 900


## 16. 为什么现在不直接做更细的 position-level update？

对于：

```text
x | a | b | y
```

理论上只修改 winner 左右的几个 pair 就够了。

但：

```text
a | a | a | a
```

会带来 repeated / overlapping pair，同时还要处理：

- non-overlapping merge；
- 一个 sequence 中 winner 多次出现；
- frequency；
- sequence collision；
- 反向索引更新。

所以我们按层次优化：

```text
Level 1：每轮全局重算
↓
Level 2：只重算受影响 sequence  ← 当前
↓
Level 3：只维护受影响的具体位置
```

只有 profile 证明 Level 2 仍然不够，才值得支付 Level 3 的复杂度成本。

## 17. 为什么现在也不急着用 heap？

当前选 winner 要扫描所有 distinct pair，约为 $O(P)$。

heap 看起来可以更快，但 BPE 每轮会修改 pair frequency。Python `heapq` 没有直接的 priority update。

常见办法是：

```text
frequency 改变
→ push 新版本
→ 旧版本暂时留在 heap
→ pop 时检查是否过期
→ 过期则丢弃
```

这叫 **lazy invalidation（延迟失效）**。

还必须严格保留 tie-break：

```text
frequency 最大
频率相同时 raw bytes pair 字典序最大
```

所以 heap 不是免费的复杂度改进。先重新 profile，再决定是否需要它。

## 18. 我们其实是在“用内存换计算”

慢版本主要保存 `pretoken_counts`。

增量版额外保存：

```text
pair_counts
pair_to_sequences
```

也就是说：

$$
\boxed{更多 metadata 内存}
\longleftrightarrow
\boxed{更少重复计算}
$$

这类 memory-compute trade-off 以后在 CS336 会不断出现，例如 KV cache、optimizer state 等。

## 19. Amdahl's Law：为什么必须先 profile？

假设总时间中：

```text
80% = merge loop
20% = 其他工作
```

即使 merge loop 无限快，剩余 20% 仍然存在，所以整体最多约 5×。

公式：

$$
S=
\frac{1}{
(1-p)+p/s
}
$$

- $p$：被优化部分原来占总时间的比例；
- $s$：该部分自身加速倍数；
- $S$：整体加速倍数。

因此优化一个只占 1% 时间的函数，即使让它快 100×，整体收益也可能很小。

In [13]:
def amdahl_speedup(p, s):
    return 1.0 / ((1.0 - p) + p / s)


for s in [2, 5, 10, 100]:
    print(f"占原时间 80% 的部分加速 {s:>3}x → 整体约 {amdahl_speedup(0.8, s):.2f}x")


占原时间 80% 的部分加速   2x → 整体约 1.67x
占原时间 80% 的部分加速   5x → 整体约 2.78x
占原时间 80% 的部分加速  10x → 整体约 3.57x
占原时间 80% 的部分加速 100x → 整体约 4.81x


## 20. 随机差分测试

一个 benchmark corpus 不够。

随机生成很多小型 `PretokenCounts`，每次都让：

```text
reference
vs
incremental
```

比较 ordered merges 和 final state。

这可以自动覆盖 repeated bytes、ties、overlapping pairs、不同 frequency 和 sequence collision 等情况。

In [14]:
def randomized_differential_test(trials=30, seed=2026):
    rng = random.Random(seed)
    alphabet = [b"a", b"b", b"c", b"d"]

    for trial in range(trials):
        counts: PretokenCounts = Counter()
        target = rng.randint(3, 12)

        while len(counts) < target:
            length = rng.randint(1, 7)
            sequence = tuple(rng.choice(alphabet) for _ in range(length))
            counts[sequence] += rng.randint(1, 10)

        num_merges = rng.randint(1, 10)

        ref_m, ref_s = train_reference(counts, num_merges)
        inc_m, inc_s, _ = train_incremental(
            counts, num_merges, validate_every_round=True
        )

        assert inc_m == ref_m, f"trial {trial}: merge 不一致"
        assert inc_s == ref_s, f"trial {trial}: state 不一致"

    print(f"{trials} 个随机差分测试全部通过")


randomized_differential_test()

30 个随机差分测试全部通过


## 21. Final Validator

正确性用硬 assertion；性能只报告 benchmark 数字，不写：

```python
assert optimized_time < reference_time
```

因为机器负载会造成时间波动。

In [15]:
def validate_optimized_bpe():
    repeated = (b"a", b"a", b"a", b"a")

    # sequence 内 (a,a) 有 3 个相邻 occurrence
    assert local_pair_counts(repeated)[(b"a", b"a")] == 3

    # corpus frequency = 7，所以全局贡献 = 3 × 7
    weighted = Counter({repeated: 7})
    assert count_adjacent_pairs(weighted)[(b"a", b"a")] == 21

    # index 只记录“这个 sequence 会受影响”
    assert build_pair_index(weighted)[(b"a", b"a")] == {repeated}

    # 连续多轮检查增量 metadata
    state = initialize_incremental_state(benchmark_counts)
    for _ in range(15):
        pair = select_best_pair(state.pair_counts)
        if pair is None:
            break
        apply_merge_incremental(state, pair)
        validate_state(state)

    # 完整差分比较
    ref_m, ref_s = train_reference(benchmark_counts, 30)
    inc_m, inc_s, _ = train_incremental(benchmark_counts, 30)

    assert inc_m == ref_m
    assert inc_s == ref_s

    print("所有 BPE 增量优化 sanity checks 通过")


validate_optimized_bpe()

所有 BPE 增量优化 sanity checks 通过
